## Using Test split for Gemini Benchmark

In [1]:
#import the labels

import pandas as pd
import time
from pathlib import Path
from google import genai

LABELS = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

VALID_LABELS = {
    "negative",
    "neutral",
    "positive"
}

In [3]:
#load only the test data
test_df = pd.read_csv("/workspaces/Financial_PhraseBank/Financial Phrasebank Project/notebooks/Financial Phrasebank Project/data/splits/test.csv")

test_df = test_df.reset_index(drop=True)

test_df["true_sentiment"] = (
    test_df["label"].map(LABELS)
)

print(test_df.shape)

(632, 4)


In [6]:
# Exact few-shot examples selected during validation.
# Keep these unchanged for the final test benchmark.
few_shot_examples = [
    ("The maritime administration said the ships had ignored warnings about the icy conditions .", "negative"),
    ("Dealers said the share was largely hit by investor disappointment about a refining margin of just 9.48 usd per barrel for the quarter and the performance of its shipping unit , which saw EBIT drop to 5 mln eur from 20 mln eur a year amid a fall in volumes and tanker rates .", "negative"),
    ("The sale of the food potato business is part of Raisio 's plan to streamline and clarify its business operations .", "neutral"),
    ("The transaction covers a building lot of 46,763 sq m and factory buildings of 19,631 sq m. Since the autumn of 2007 , when Scanfil discontinued its production operations in Oulu , the facilities have been partly leased to other parties .", "neutral"),
    ("The Finnish group anticipates a sales gain of EUR42m after tax and expenses .", "positive"),
    ("ADPnews - Jun 30 , 2009 - Finnish stainless steel maker Outokumpu Oyj HEL : OUT1V said yesterday it signed a three-year EUR 900 million USD 1.3 bn revolving credit facility .", "positive"),
]
print("Few-shot examples:", len(few_shot_examples))

Few-shot examples: 6


In [7]:
#prompt use from validation run
def build_few_shot_prompt(sentence):
    examples_text = "\n\n".join(
        f"Sentence: {example}\nLabel: {label}"
        for example, label in few_shot_examples
    )

    return f"""
You are performing financial sentiment classification.

Classify financial news from the perspective of an investor.

Use exactly one label:

negative = financially unfavorable implications
neutral = no clear positive or negative financial implication
positive = financially favorable implications

Examples:

{examples_text}

Now classify this sentence.

Sentence:
{sentence}

Return only one label: negative, neutral, or positive.
""".strip()

In [8]:
!pip install -q -U google-genai


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [9]:
#initialize gemini
from google import genai

client = genai.Client()

MODEL_NAME = "gemini-3.5-flash-lite"

In [15]:
#classifier helper function
import re

def classify_with_gemini_few_shot(sentence):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_few_shot_prompt(sentence)
    )

    raw_prediction = response.text.strip().lower()

    # Ideal case
    if raw_prediction in VALID_LABELS:
        return raw_prediction

    # Handle outputs such as:
    # "label: neutral"
    # "the label is positive"
    matches = re.findall(
        r"\b(negative|neutral|positive)\b",
        raw_prediction
    )

    unique_matches = set(matches)

    if len(unique_matches) == 1:
        return matches[0]

    raise ValueError(
        f"Unexpected Gemini output: {raw_prediction}"
    )

In [16]:
#sanity check before 632 sentence call
print(test_df.loc[0, "sentence"])

test_prediction = classify_with_gemini_few_shot(
    test_df.loc[0, "sentence"]
)

print("Gemini prediction:", test_prediction)
print("True sentiment:", test_df.loc[0, "true_sentiment"])

Subject-matter of the invention furthermore is the use of the cyclone for separating partly molten particles . ''
Gemini prediction: neutral
True sentiment: neutral


In [22]:
#Run the final 632-row benchmark with a resumable loop so Codespaces/API failures cannot wipe out our progress
import time
import pandas as pd
from pathlib import Path

progress_path = Path("gemini_test_progress.csv")

# Resume if progress already exists
if progress_path.exists():
    progress_df = pd.read_csv(progress_path)
    test_predictions = progress_df["prediction"].tolist()

    print(
        f"Resuming from {len(test_predictions)} saved predictions."
    )
else:
    test_predictions = []
    print("Starting final Gemini test benchmark.")

start_index = len(test_predictions)

for i in range(start_index, len(test_df)):

    sentence = test_df.loc[i, "sentence"]

    print(f"Classifying {i + 1}/{len(test_df)}...")

    success = False

    for attempt in range(5):

        try:
            prediction = classify_with_gemini_few_shot(
                sentence
            )

            test_predictions.append(prediction)

            success = True
            break

        except Exception as e:

            message = str(e)

            print(
                f"Attempt {attempt + 1} failed."
            )

            if (
                "429" in message
                or "RESOURCE_EXHAUSTED" in message
            ):
                wait_time = 30 * (attempt + 1)

            elif (
                "503" in message
                or "UNAVAILABLE" in message
            ):
                wait_time = 2 ** attempt

            else:
                print(message)
                break

            print(
                f"Waiting {wait_time} seconds..."
            )

            time.sleep(wait_time)

    if not success:
        print(
            f"Stopping safely at sentence {i + 1}."
        )
        break

    # Save after EVERY successful call
    pd.DataFrame({
        "prediction": test_predictions
    }).to_csv(
        progress_path,
        index=False
    )

    # Avoid hammering the API
    time.sleep(3)

Resuming from 384 saved predictions.
Classifying 385/632...
Attempt 1 failed.
Waiting 30 seconds...
Attempt 2 failed.
Waiting 60 seconds...
Attempt 3 failed.
Waiting 90 seconds...
Attempt 4 failed.
Waiting 120 seconds...
Attempt 5 failed.
Waiting 150 seconds...
Stopping safely at sentence 385.


In [18]:
#check finished proccess
print("Predictions completed:", len(test_predictions))

Predictions completed: 138


In [ ]:
#save the complete final result
test_df["gemini_prediction"] = test_predictions

test_df.to_csv(
    "gemini_final_test_results.csv",
    index=False
)

print("Final Gemini test results saved.")

ValueError: Length of values (56) does not match length of index (632)

In [ ]:
#calculate final metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_true = test_df["true_sentiment"]
y_pred = test_df["gemini_prediction"]

gemini_test_accuracy = accuracy_score(
    y_true,
    y_pred
)

gemini_test_macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro"
)

print(
    "Gemini Test Accuracy:",
    gemini_test_accuracy
)

print(
    "Gemini Test Macro-F1:",
    gemini_test_macro_f1
)

print(
    classification_report(
        y_true,
        y_pred,
        labels=[
            "negative",
            "neutral",
            "positive"
        ],
        digits=3
    )
)

In [ ]:
#compute confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

labels = [
    "negative",
    "neutral",
    "positive"
]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=labels
)

print(cm)

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
).plot(ax=ax)

ax.set_title(
    "Gemini 3.5 Flash-Lite Few-Shot — Test Set"
)

plt.show()